Langchain 기초

In [6]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Tue" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W4" / "Tue"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\hanwha-agent


In [7]:
from langchain_anthropic import ChatAnthropic
from app.core.config import get_settings

llm=ChatAnthropic(
    model='claude-haiku-4-5',
    max_tokens=500,
)

response=llm.invoke(
    'RAG를 한 문장으로 설명하시오'
)

print(response)

content='# RAG (Retrieval-Augmented Generation)\n\n**외부 데이터베이스에서 관련 정보를 검색하여 LLM의 답변 생성을 보강하는 기술입니다.**' additional_kwargs={} response_metadata={'id': 'msg_011CfHzf4Bscf1WF6kYi35fP', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 23, 'output_tokens': 65, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a0c7c6-5917-7072-87c9-26e840188659-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 65, 'total_tokens': 88, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}


In [8]:
llm.invoke(
    'RAG가 무엇인지 비전공자에게 설명해주'
)
# RAG / Embedding / Vector DB의 개념 등

AIMessage(content='# RAG(검색증강생성)를 쉽게 설명해드릴게요\n\n## 핵심 아이디어: AI의 "찾아보기" 기능\n\n**RAG = 검색 + AI 답변**\n\n일반 AI와의 차이를 예로 들면:\n\n### 🤖 일반 ChatGPT\n- "삼성전자 2024년 4분기 실적이 뭐야?"\n- → 학습 데이터에만 의존해서 답변 (최신 정보 없음, 틀릴 수 있음)\n\n### 🔍 RAG 활용 AI\n- "삼성전자 2024년 4분기 실적이 뭐야?"\n- → 먼저 **인터넷/문서에서 검색** → 최신 정보 찾음 → 답변 제공\n- → 더 정확하고 최신의 답변 가능\n\n## 간단한 비유\n\n마치 학생이 시험 볼 때:\n- **일반 AI**: 머릿속에만 있는 지식으로 답하기\n- **RAG**: 교과서를 찾아보며 정확한 답 확인하기 ✅\n\n## 실제 쓰임새\n- 기업 내부 문서 질문 (사내 규정, 매뉴얼 등)\n- 최신 뉴스 기반 챗봇\n- 의료 기록 조회\n- 고객지원 챗봇\n\n**장점**: 더 정확하고 신뢰할 수 있는 답변 제공!\n\n이해 안 되는 부분이 있으면 더 쉽게 설명해드릴게요😊', additional_kwargs={}, response_metadata={'id': 'msg_011CfHzf9JwrxXmfdUob9Hkj', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 28, 'output_

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트 생성
prompt = ChatPromptTemplate.from_template(
	"{topic}에 대해 비전공자도 이해할 수 있도록 쉽게 설명 plz"
)

# 2. LLM 생성
llm=ChatAnthropic(
    model='claude-haiku-4-5',
    max_tokens=500,
)

# 2와 3 사이. AIMessage에서 문자열 추출을 위한 parser
parser=StrOutputParser()

# 3. 프롬프트와 LLM 연결
chain=prompt|llm|parser

# 4. chain 실행
response=chain.invoke({
    'topic':'RAG'
})

# 5. Claude 답변 출력
# print(response.content)
print(response)
print(type(response))

# RAG(Retrieval-Augmented Generation) 쉽게 설명 🎯

## 🍕 피자 배달 앱으로 비유하면

**RAG가 없는 경우:**
- ChatGPT에게 "서울 강남역 근처 피자집 추천해줘"라고 물어봄
- AI가 학습 데이터(2년 전 정보)로만 대답
- "폐점된 가게", "없는 주소" 추천 → **정보가 낡고 부정확함** ❌

**RAG가 있는 경우:**
- AI가 먼저 **현재 배달앱 DB를 검색**해서 정보 수집
- 최신 정보를 기반으로 답변
- "강남역 인근 평점 4.8 피자집 3곳" 추천 ✅

---

## 📚 도서관 사서로 비유하면

- **일반 AI** = 머리 좋지만 책을 안 본 사람
  - "제 기억으로는 이렇습니다" (틀릴 수 있음)

- **RAG AI** = 도서관 사서
  - 먼저 책장에서 관련 책 찾음 (검색)
  - 그 책 내용을 읽고 답변함 (생성)
  - 더 정확하고 최신 정보 제공 ✨

---

## 🔄 RAG의 3가지 단계

```
1️⃣ 검색(Retrieval) → 관련 정보 찾기
   예: "강남역 피자집" 관련 자료 검색

2️⃣ 검토 →
<class 'langchain_core.messages.base.TextAccessor'>


Langchain 활용

In [10]:
# 함수 3개

def build_prompt(question:str)->str:
    return f'[규정질문]:{question}'

def call_llm(prompt:str)->str:
    return "{'answer':'출장 비용은 1일 3만원임', 'doc_id':'DOC-HR-012}"

def parser(text:str)->dict:
    import json
    return json.loads(text)

In [ ]:
from langchain_core.runnables import Runnable, RunnableLambda

# 파이프라인
step=RunnableLambda(build_prompt)
print(type(step))
# step.invoke()

<class 'langchain_core.runnables.base.RunnableLambda'>


In [13]:
pipeline=RunnableLambda(build_prompt) | call_llm | parser
print(type(pipeline))
# pipeline.invoke()

<class 'langchain_core.runnables.base.RunnableSequence'>


In [14]:
def lookup(payload:dict)->str:
    known={'DOC-HR-012','DOC-PU-007','DOC-SE-002'}
    if payload['doc_id'] not in known:
        raise ValueError('모르는 문서')
    return f'{payload['doc_id']} 조회 완료'
lookup_chain=RunnableLambda(lookup)

# 여러개 요청
inputs=[
    {'doc_id':'DOC-HR-012'},
    {'doc_id':'DOC-XX-999'},
    {'doc_id':'DOC-PU-007'},
    {'doc_id':'DOC-SE-002'},
]

try:
    # lookup_chain.invoke({'doc_id':'DOC-HR-012'}) # 한 개일 때는 이렇게 하면 되고
    lookup_chain.batch(inputs) # 여러 개일 때는 이렇게 하면 됨
except ValueError as e:
    print('실패...',e)

results=lookup_chain.batch(inputs, return_exceptions=True)
for one in results:
    print(one)

실패... 모르는 문서
DOC-HR-012 조회 완료
모르는 문서
DOC-PU-007 조회 완료
DOC-SE-002 조회 완료


In [15]:
from langchain_core.language_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser

llm=FakeListChatModel(responses=['부산 출장 비용 1일 2만원임'])
out=llm.invoke('부산 출장비 얼마')
print(out)
print(type(out))

text_chain=llm|StrOutputParser()
print(text_chain.invoke('부산 출장 일일 비용은?'))

content='부산 출장 비용 1일 2만원임' additional_kwargs={} response_metadata={} id='lc_run--01a0c7f6-5caa-7bd1-b006-80592f376c61-0' tool_calls=[] invalid_tool_calls=[]
<class 'langchain_core.messages.ai.AIMessage'>
부산 출장 비용 1일 2만원임


In [18]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

ANSWER='일일 지원 비용 2만원 - 숙박 실비'

gllm=GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
pieces=list((gllm|StrOutputParser()).stream('부산 출장 일일 비용은?'))

print('조각 개수:',len(pieces))
print(pieces)

gllm2=GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
print('이어서 출력:', end='') # 줄내림 없이 옆으로 출력
for piece in ((gllm2|StrOutputParser()).stream('부산 출장 일일 비용은?')):
    print(piece, end='', flush=True)

print()

조각 개수: 13
['일일', ' ', '지원', ' ', '비용', ' ', '2만원', ' ', '-', ' ', '숙박', ' ', '실비']
이어서 출력:일일 지원 비용 2만원 - 숙박 실비


In [19]:
from langchain_core.runnables import RunnablePassthrough

enrich=RunnablePassthrough.assign(
    doc_id=lambda d:'DOC-HR-011',
    grade=lambda d:'일반'
)
print(enrich.invoke({'question':'부산 출장 일일 비용은?'}))
print(type(enrich))

line=enrich | RunnableLambda(lambda d: f"[{d['doc_id']}/{d['grade']}] {d['question']}")
print(line.invoke({'question':'부산 출장 일일 지원 비용은?'}))

{'question': '부산 출장 일일 비용은?', 'doc_id': 'DOC-HR-011', 'grade': '일반'}
<class 'langchain_core.runnables.passthrough.RunnableAssign'>
[DOC-HR-011/일반] 부산 출장 일일 지원 비용은?


In [20]:
from langchain_core.runnables import RunnableParallel

both=RunnableParallel(
    upper=RunnableLambda(lambda  s:s.upper()),
    length=RunnableLambda(lambda s:len(s))
)
print('parallel:',both.invoke('DOC-HR-002'))

parallel: {'upper': 'DOC-HR-002', 'length': 10}
